<a href="https://colab.research.google.com/github/jasonkjw/daily_coding_commit/blob/main/%EC%98%A8%EB%8F%84_%EC%84%A0%ED%98%95%EB%B3%B4%EA%B0%84%EC%B2%98%EB%A6%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

def calculate_design_temp(file_path):
    # 1. 데이터 로드 및 전처리
    df = pd.read_csv(file_path)
    time_cols = [str(i).zfill(2) for i in range(1, 25)] # 01~24시 컬럼

    # 2. 결측치 처리 (-99.0 및 공백을 NaN으로 변환)
    for col in time_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.replace(-99.0, np.nan)

    # 3. 선형 보간 (시간별 빈틈 메우기)
    df[time_cols] = df[time_cols].interpolate(axis=1, limit_direction='both') # 행(시간) 방향
    df[time_cols] = df[time_cols].interpolate(axis=0, limit_direction='both') # 열(날짜) 방향

    # 4. 전체 시간 기온을 1차원 배열로 평탄화
    all_temps = df[time_cols].values.flatten()
    all_temps = all_temps[~np.isnan(all_temps)] # 남아있는 결측치 제거

    # 5. ASHRAE 0.4% 산출 (상위 0.4% 지점인 99.6 백분위수)
    design_temp = np.percentile(all_temps, 99.6)
    return round(design_temp, 2)

# 결과 출력 예시
print(f"부산(159) 설계 온도: {calculate_design_temp('Busan_STN_DATA.csv')}°C")